In [0]:
import os
from datetime import datetime
import pandas as pd


def calculate_care_gaps(
    member_df: pd.DataFrame,
    claims_df: pd.DataFrame,
    lookup_df: pd.DataFrame,
    measurement_year: int = 2026,
) -> pd.DataFrame:
    if member_df.empty or lookup_df.empty:
        return pd.DataFrame()

    # -------------------------------------------------------------
    # 1. Lowercase all column names (e.g. ServiceStartDate -> servicestartdate)
    # -------------------------------------------------------------
    member = member_df.copy()
    member.columns = member.columns.str.strip().str.lower()

    claims = claims_df.copy()
    if not claims.empty:
        claims.columns = claims.columns.str.strip().str.lower()

    lookup = lookup_df.copy()
    lookup.columns = lookup.columns.str.strip().str.lower()

    # -------------------------------------------------------------
    # 2. Convert Lookup Dates & Codes
    # -------------------------------------------------------------
    if "servicestartdate" in lookup.columns:
        lookup["servicestartdate"] = pd.to_datetime(lookup["servicestartdate"])
    else:
        lookup["servicestartdate"] = pd.to_datetime(f"{measurement_year}-01-01")

    if "serviceenddate" in lookup.columns:
        lookup["serviceenddate"] = pd.to_datetime(lookup["serviceenddate"])
    else:
        lookup["serviceenddate"] = pd.to_datetime(f"{measurement_year}-12-31")

    lookup["code"] = lookup["code"].astype(str).str.strip()

    # -------------------------------------------------------------
    # 3. Process Member & Claims
    # -------------------------------------------------------------
    if "dateofbirth" in member.columns:
        member["dob"] = pd.to_datetime(member["dateofbirth"])
    elif "dob" in member.columns:
        member["dob"] = pd.to_datetime(member["dob"])
    elif "patient_dob" in member.columns:
        member["dob"] = pd.to_datetime(member["patient_dob"])

    member_id_col = (
        "uniquepersonkey"
        if "uniquepersonkey" in member.columns
        else ("patient_id" if "patient_id" in member.columns else "member_id")
    )

    if not claims.empty:
        claims["procedure_code"] = claims["procedure_code"].astype(str).str.strip()
        claims["line_service_date"] = pd.to_datetime(claims["line_service_date"])

        # Match procedure_code from CSV with code from lookup
        matched_claims = claims.merge(
            lookup,
            left_on="procedure_code",
            right_on="code",
            how="inner",
        )

        valid_claims = matched_claims[
            (matched_claims["line_service_date"] >= matched_claims["servicestartdate"])
            & (matched_claims["line_service_date"] <= matched_claims["serviceenddate"])
        ]
    else:
        valid_claims = pd.DataFrame()

    # -------------------------------------------------------------
    # 4. Evaluate Care Gaps
    # -------------------------------------------------------------
    unique_measures = (
        lookup[["measurename", "measuresource"]]
        .drop_duplicates()
        .to_dict(orient="records")
    )

    all_results = []
    year_end = pd.to_datetime(f"{measurement_year}-12-31")

    for target in unique_measures:
        m_name = target["measurename"]
        m_source = target["measuresource"]

        rules = lookup[
            (lookup["measurename"] == m_name)
            & (lookup["measuresource"] == m_source)
        ]

        if rules.empty:
            continue

        rule_gender = rules["gender"].iloc[0]
        min_age = rules["minage"].iloc[0]
        max_age = rules["maxage"].iloc[0]

        temp_member = member.copy()
        temp_member["calculated_age"] = (
            year_end - temp_member["dob"]
        ).dt.days // 365.25

        # Denominator Filter
        gender_mask = (
            (temp_member["gender"] == rule_gender)
            if rule_gender in ["M", "F"]
            else True
        )
        denom_df = temp_member[
            gender_mask
            & (temp_member["calculated_age"] >= min_age)
            & (temp_member["calculated_age"] <= max_age)
        ].copy()

        if denom_df.empty:
            continue

        # Numerator Check
        if not valid_claims.empty:
            num_members = valid_claims[
                (valid_claims["measurename"] == m_name)
                & (valid_claims["measuresource"] == m_source)
            ]["patient_id"].unique()
        else:
            num_members = []

        denom_df["measureName"] = m_name
        denom_df["measureSource"] = m_source
        denom_df["care_gap_status"] = denom_df[member_id_col].apply(
            lambda m_id: "CLOSED" if m_id in num_members else "OPEN"
        )
        denom_df["evaluation_date"] = datetime.today().strftime("%Y-%m-%d")

        all_results.append(denom_df)

    return (
        pd.concat(all_results, ignore_index=True)
        if all_results
        else pd.DataFrame()
    )